# GNSS Station Discovery and Retrieval

**Prerequisites:** Working knowledge of Python and Jupyter notebooks, understanding of GNSS data

**GeoLab compute:** Default Image (4 GB RAM, ~0.5 CPU)

## Overview

This notebook demonstrates how to retrieve processed GNSS position time series data from the GAGE web services API provided by EarthScope. You will define a geographic study area, identify available stations, visualize them on an interactive map, and download position time series data to your scratch storage. The output can be used to quickly visualize the GNSS stations available in your Area of Interest (AOI) and download the position time series which can then be used for future workflows.



## Learning Objectives

By the end of this notebook you will be able to:

1. Query the GAGE API to find GNSS stations within a bounding box
2. Visualize station locations on an interactive map
3. Retrieve and save processed position time series data for multiple stations
4. Understand the structure of the GAGE GeoCSV response format

## Related Documentation

- [GAGE Web Services Documentation](https://www.unavco.org/data/web-services/documentation/documentation.html#/GNSS47GPS)
- [GNSS Position Data Documentation](https://www.unavco.org/data/web-services/documentation/gps-position-documentation.html)

## Setup
We begin by importing the necessary Python libraries and setting the base URL for all API requests.

In [ ]:
import os
import shutil
import requests
import pandas as pd
import folium
import matplotlib.pyplot as plt
from io import StringIO

In [ ]:
#base URL for GAGE/UNAVCO web service requests.
BASE_URL = "https://web-services.unavco.org"

## Defining Study Area

We define our study area using a rectangular bounding box that is defined by minimum and maximum latitude and longitude (in degrees). We also define the time range we are interested in using the parameters `START` and `END` in the format `YYYY-MM-DD`.

In [ ]:
MINLAT, MAXLAT = 43, 46
MINLON, MAXLON = -125, -123
START, END = "2017-01-01", "2024-01-01"

Before retrieving data, we visualize the bounding box on an interactive map to make sure it covers the area we intend. This is a good sanity check to see our coordinates match our intended AOI before making API calls.

In [ ]:
def plot_bbox(minlat, maxlat, minlon, maxlon, height=400):
    center_lat = (minlat + maxlat) / 2
    center_lon = (minlon + maxlon) / 2

    fig = folium.Figure(height=height)
    m = folium.Map(location=[center_lat, center_lon], zoom_start=7,
                   tiles="CartoDB positron")
    m.add_to(fig)

    folium.Rectangle(
        bounds=[[minlat, minlon], [maxlat, maxlon]],
        color="blue",
        fill=True,
        fill_opacity=0.1,
        tooltip=f"Bounding box: ({minlat}, {minlon}) to ({maxlat}, {maxlon})"
    ).add_to(m)

    return fig

In [ ]:
plot_bbox(MINLAT, MAXLAT, MINLON, MAXLON)

## Retrieving Station Metadata

The GAGE API endpoint `gps/metadata/sites/v1` retrieves site metadata for all GNSS sites that fall within a spatial bounding box defined by north and south latitude, and east and west longitude. The response is a GeoCSV string (a comma separated text format that contains geographic data). Alternatively, we can also fetch the request in a json or XML format.

Let's first fetch the raw response and inspect it before parsing.

In [ ]:
def get_stations_in_bbox(minlat, maxlat, minlon, maxlon):
    params = {
        "minlatitude": minlat,
        "maxlatitude": maxlat,
        "minlongitude": minlon,
        "maxlongitude": maxlon,
        "format": "csv"
    }
    r = requests.get(f"{BASE_URL}/gps/metadata/sites/v1", params=params) 
    r.raise_for_status()
    return r.text

`requests.get()` returns the entire HTTP response body as a single Python string. We can inspect the raw output as below:

In [ ]:
meta_csv = get_stations_in_bbox(MINLAT, MAXLAT, MINLON, MAXLON)
for line in meta_csv.splitlines()[:3]:
    print(line+"\n")

The first line is the `#fields=` header that encodes column names and types. The following rows after the header row are data. Notice `CORV` appears twice with different receiver types, confirming that stations have multiple session records.


## Parsing Station Metadata

The raw GeoCSV response contains column names encoded in the `#fields=` comment line with type annotations like `[type='string']` that we need to strip out. Each station also appears multiple times in the response — once per equipment configuration (antenna/receiver changes over time). We deduplicate by station ID for getting the station list and optionally filter by our time range of interest.

In [ ]:
def parse_station_metadata(csv_text, start=None, end=None):
    # extract column names from the #fields= comment line
    fields_line = [l for l in csv_text.splitlines() if l.startswith('#fields=')][0]
    col_names = [f.split('[')[0] for f in fields_line.replace('#fields=', '').split(',')]

    # parse data rows (skip all comment lines)
    lines = [l for l in csv_text.splitlines() if not l.startswith('#')]
    df = pd.read_csv(StringIO("\n".join(lines)), header=None)
    df.columns = col_names

    # convert session times to datetime
    df["session_start_time"] = pd.to_datetime(df["session_start_time"])
    df["session_stop_time"] = pd.to_datetime(df["session_stop_time"])

    # filter to stations active during our time range
    # a station is included if it has any overlap with [start, end]
    if start:
        df = df[df["session_stop_time"] >= pd.to_datetime(start, utc=True)]
    if end:
        df = df[df["session_start_time"] <= pd.to_datetime(end, utc=True)]

    # one row per station
    df = df.drop_duplicates(subset="ID")

    return df[["ID", "latitude", "longitude"]]

We now parse the raw response and filter to stations that have data during our time range. The result is one row per station with its ID and coordinates.

In [ ]:
stations = parse_station_metadata(meta_csv, start=START, end=END)
print(f"Found {len(stations)} stations")
stations.reset_index(drop=True).head()

## Visualizing Stations on a Map

Now that we have our station list, we can add them to the map alongside our bounding box to give us a complete picture of our study area i.e. where the bbox is geographically and which stations fall within it.

In [ ]:
def plot_stations_map(stations_df, minlat, maxlat, minlon, maxlon, height=500):
    center_lat = (minlat + maxlat) / 2
    center_lon = (minlon + maxlon) / 2

    fig = folium.Figure(height=height)
    m = folium.Map(location=[center_lat, center_lon], zoom_start=7,
                   tiles="CartoDB positron")
    m.add_to(fig)

    # bounding box
    folium.Rectangle(
        bounds=[[minlat, minlon], [maxlat, maxlon]],
        color="blue",
        fill=True,
        fill_opacity=0.1,
        tooltip=f"Bounding box: ({minlat}, {minlon}) to ({maxlat}, {maxlon})"
    ).add_to(m)

    # station markers
    for _, row in stations_df.iterrows():
        folium.RegularPolygonMarker(
            location=[row["latitude"], row["longitude"]],
            number_of_sides=3,
            radius=5,
            rotation=30,
            color="red",
            fill=True,
            fill_opacity=1,
            popup=row["ID"],
            tooltip="Station: "+row["ID"]
        ).add_to(m)

    return fig

In [ ]:
plot_stations_map(stations, MINLAT, MAXLAT, MINLON, MAXLON)

You should see red triangle markers within the blue bounding box. Click or hover over a marker to see the station ID. If no stations appear, your bounding box may be too small or your date range too restrictive.

## Retrieving Position Time Series

Now that we have our station list and have verified the locations on the map, we can fetch the processed position time series for each station from the GAGE API endpoint `/gps/data/position/{station}/v3`.

The API returns a response with north, east, and up displacement offsets relative to a reference coordinate at each station. The API parameters we use are:

- **analysisCenter:** GNSS time series position solutions are available from four different analysis centers. CWU (default), NMT, PBO and UNR.
- **referenceFrame:** The position solutions are available in different reference frames depending on the analysis center. `nam14` (North America fixed) is the default whereas the global frame `igs14` is available from all analysis centers.
- **report:** controls the output content:
  - `short` - (default) north/east/up offsets and standard deviations only
  - `long` - full source file including cartesian and geodetic coordinates and all associated error estimates and covariances
- **dataPostProcessing:** post-processing applied after data retrieval:
  - `Uncleaned` (default) — data returned as-is, no post-processing
  - `Cleaned` — offset values for North, East, or Up are set to NULL when the standard deviation exceeds 20mm
- **format:**  Response formats. CSV, JSON or XML.

Data is always fetched fresh from the API and saved to your scratch storage.

In [ ]:
SAVE_DIR = os.path.join(os.environ["SCRATCH_BUCKET"], "gnss_positions")
shutil.rmtree(SAVE_DIR) if os.path.exists(SAVE_DIR) else None

def get_position_timeseries(station_code, start=None, end=None):
    params = {
        "analysisCenter": "cwu",
        "referenceFrame": "nam14",
        "report": "short",
        "dataPostProcessing": "Cleaned",
        "format": "csv"
    }
    if start:
        params["starttime"] = start
    if end:
        params["endtime"] = end

    r = requests.get(f"{BASE_URL}/gps/data/position/{station_code}/v3", params=params)
    r.raise_for_status()
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    filepath = os.path.join(SAVE_DIR, f"{station_code}.csv")
    with open(filepath, "w") as f:
        f.write(r.text)
    # print(f"  Saved to {filepath}")
    return r.text


def parse_position_csv(csv_text):
    lines = [l for l in csv_text.splitlines() if not l.startswith('#')]
    df = pd.read_csv(StringIO("\n".join(lines)))
    df.columns = df.columns.str.strip()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    return df

Before fetching all stations, we test on a single station to verify the response format and confirm our parameters are working correctly.

In [ ]:
# fetch and inspect a single station before running all stations
code = stations["ID"].iloc[0]
print(f"Fetching {code}...")
pos_csv = get_position_timeseries(code, start=START, end=END)
df = parse_position_csv(pos_csv)
print(f"{len(df)} epochs")
df.head()

You should see a DataFrame with columns: `Datetime`, `delta N`, `delta E`, `delta U`, and associated standard deviations. Each row is one daily position estimate. 

## Retrieving Data for All Stations

Now we fetch position time series for all stations. Stations that return a 404 error will be skipped automatically. See the Troubleshooting section for guidance.

In [ ]:
for _, row in stations.iterrows():
    code = row["ID"]
    print(f"Fetching {code}...")
    try:
        pos_csv = get_position_timeseries(code, start=START, end=END)
        df = parse_position_csv(pos_csv)
        print(f"  {len(df)} epochs")
    except requests.HTTPError as e:
        print(f"  Skipping {code}: {e}")

## Summary

Let's print a summary of what was retrieved to confirm everything looks as expected 
before moving on to analysis.

In [ ]:
successful = []
failed = []

for _, row in stations.iterrows():
    code = row["ID"]
    filepath = os.path.join(SAVE_DIR, f"{code}.csv")
    if os.path.exists(filepath):
        successful.append(code)
    else:
        failed.append(code)

print(f"Total stations found:      {len(stations)}")
print(f"Successfully retrieved:    {len(successful)}")
print(f"Failed:                    {len(failed)}")

if failed:
    print(f"\nFailed stations: {failed}")

print(f"\nData saved to: {SAVE_DIR}")

## Troubleshooting

- If a station returns a 404 error, for example `Skipping station_ID: 404 Client Error` it exists in the metadata but has no processed position solution for your date range. This is normal and the station will be skipped automatically. One solution might be to change the analysis center, for example from `cwu` to `unr` to see if another analysis center has processed the data.

## Try It Yourself

- Change `MINLAT`, `MAXLAT`, `MINLON`, `MAXLON` to a different region and re-run
- Change `START`, `END` to a different time range and re-run
- Change `report` from `short` to `long`. What additional columns appear?